<a href="https://colab.research.google.com/github/Deepthi-Bhargavi-Kasturi/ai-learning-journey/blob/master/textgeneration_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
#Install three essential python libraries created by HuggingFace for building, training and testing models
#transaformers: core library used to download, run and finetune state-of-the-art pre-trained AI models
#datasets: this library manages data needed to train/fine-tune or test the models
#evaluate: this helps evaluate the accuracy of the model's output - how well your model is performing.
!pip install transformers datasets evaluate

#LOAD ELI5 DATASET
#load_dataset is a function used to load a specific category of dataset
#It downloads and caches the data of the specified category.
from datasets import load_dataset

#dany0407/eli5_category:
#unique address to the dataset. <username> / <repository name>
#username is who uploaded or mirrorred this data set to HuggingFace and eli5_category is the repository : name of the dataset itself.
#This tells the data is from Reddit "Explain Like I'm Five"

#split="train[:5000]"
#We are loading first 5000 examples instead of entire dataset when starting with the fine-tuning
#train here indicates that we are accessing training data (not testing data) and want to partition training data
#train[:5000] - python slice method. Starting from index 0 to up to, but not including data at index 5000
#Examples: train[5000:10000] grabs the next 5000 rows from 5000 to 10000
#          train[-1000:] grabs only the last 1000 rows of the training set
#          validation[:10%] grabs the first 10% of the validation dataset.
# training data vs validation data set - after studying the training data, it tries to answer from the validation data set to measure how well the
# model is performing that is how well the model understood the concepts or if it just memorized the concepts
eli5 = load_dataset("dany0407/eli5_category", split="train[:5000]")

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert/distilgpt2")


#the fetched first 5000 rows data set is divided into to train and test data sets
#train_test_split is a built in hugging face method - randomly shuffles the data and divides into 2 sections (train data, test data)
#test_size=0.2 -  test data is 20% and so train data will be 80% from 5000 rows (100%)
#the original single data set eli5 is now overridden by grouped dictionary structure containing both splits - train and test
#output looks like:
#   DatasetDict({
#     train: Dataset({
#          features: ['title', 'text', 'category'],
#          num_rows: 4000
#     }),
#     test: Dataset({
#           features: ['title', 'text', 'category'],
#           num_rows: 1000
#      })
#   })
#how to access data:
#   training_data = eli5["train"]
#   test_data = eli5["test"]
#   eli5["train"][0] - every single cell from the first row -- see the output it is entire one row.
eli5 = eli5.train_test_split(test_size=0.2)
print(eli5["train"][0])

#preprocessing - step 1
#flatten
eli5 = eli5.flatten()
print(eli5["train"][0])

#tokenize list of strings in the answers.text field using tokenizer() function.
#for this write a function. The parameter is the elements of eli5["train"] array that is at index 0, 1 etc
def preprocess_function(examples):
  return tokenizer([" ".join(x) for x in examples["answers.text"]])

#apply this function to entire dataset that is eli5["train"]
#for this use Datasets map method - loops over every single row / element in the dataset
#error at remove_columns = eli5.column_names: ValueError: Column to remove ['train', 'test'] not in the dataset. Current columns in the dataset: ['q_id', 'title', 'selftext', 'category', 'subreddit', 'answers.a_id', 'answers.text', 'answers.score', 'answers.text_urls', 'title_urls', 'selftext_urls']
#batched = True -> this is to process multiple elements of datasets at once by grouping / batching them together. 1000 by default
#num_proc = 4 -> increase number of processes - tells the computer that we need 4 CPU porcessing cores to work on this task at the exact same time. It divides dataset into 4 equal pieces, works on them / runs them in parallel and stictches them back together
#remove_columns = eli5["train"].column_names -> err eli5.column_names -> eli5["train"].column_names: because we have split the data set into train and test #removes old text columns and keep only tokenized outputs
tokenized_eli5 = eli5.map(
    preprocess_function,
    batched = True,
    num_proc = 4,
    remove_columns = eli5["train"].column_names

)
#output at this step -> error -> [transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1109 > 1024). Running this sequence through the model will result in indexing errors
#Datasets comtains token sequences, some of these are longer than the specified maximum sequence length for this model (1109 > 1024).

#To solve this we use second preprocessing function
block_size = 128
def group_texts(examples):
  concatenated_examples = {
      k : sum(examples[k],[]) for k in examples.keys() #flattening of lists
  }

  total_length = len(concatenated_examples[list(examples.keys())[0]])
  if total_length >= block_size:
    total_length = (total_length // block_size)*block_size

  result = {
            k:[t[i:i+block_size] for i in range(0,total_length,block_size)]
            for k,t in concatenated_examples.items()
  }

  result["labels"] = result["input_ids"].copy()
  return result

#Apply above function over the entire dataset
lm_dataset = tokenized_eli5.map(group_texts, batched = True, num_proc = 4)

print(len(lm_dataset["train"][0]["input_ids"]))
#Output: 128

print(lm_dataset["train"][0]["input_ids"])
#Output: [2504, 338, 1444, 6588, 46314, 1358, 11, 290, 340, 318, 257, 1517, 356, 460, 466, 13, 1318, 561, 761, 284, 307, 281, 12964, 1581, 44, 20958, 2033, 286, 340, 284, 3753, 703, 881, 356, 821, 5137, 656, 262, 8137, 13, 632, 338, 4577, 284, 4646, 262, 6588, 5072, 13, 632, 561, 307, 588, 2111, 284, 12051, 5789, 748, 282, 1883, 6134, 284, 787, 4713, 1660, 780, 286, 674, 3236, 3512, 329, 10150, 75, 485, 14860, 13, 775, 460, 11, 475, 356, 6584, 470, 13, 383, 3037, 318, 44192, 1091, 276, 11, 475, 772, 611, 340, 547, 20823, 4166, 340, 561, 307, 2089, 284, 3494, 340, 13, 775, 10385, 11863, 290, 17173, 7718, 23461, 284, 7375, 17, 290, 1660, 284, 787, 2568, 13, 37941, 9482, 4898, 11, 33512, 11]

#DataCollation
from transformers import DataCollatorForLangugaeModeling
tokenizer.pad_token = tokenizer.eos_token
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)



{'q_id': '74ovhf', 'title': "If plants can convert Carbon Dioxide into Oxygen, why can't we create machines to do the same thing to control the amount of CO2 in the environment?", 'selftext': '', 'category': 'Chemistry', 'subreddit': 'explainlikeimfive', 'answers': {'a_id': ['dnzyjem', 'do01npc', 'do029vu', 'do005nd', 'do02l7c'], 'text': ["That's called carbon sequestration, and it is a thing we can do. There would need to be an ENORMOUS amount of it to counter how much we're putting into the atmosphere. It's easier to reduce the carbon output. It would be like trying to justify expensive desalination plants to make fresh water because of our huge demand for waterslide parks.", "We can, but we shouldn't. The technology is undeveloped, but even if it were extensively developed it would be bad to implement it. We convert oxygen and hydrocarbons to CO2 and water to make energy. ie burning wood, charcoal, coal, natural gas, and liquid oil products. Forgetting about the issue of scale-able 

Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1521 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4145 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1385 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (3313 > 1024). Running this sequence through the model will result in indexing errors


Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1924 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1202 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1079 > 1024). Running this sequence through the model will result in indexing errors
[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1145 > 1024). Running this sequence through the model will result in indexing errors


Map (num_proc=4):   0%|          | 0/4000 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/1000 [00:00<?, ? examples/s]

128
[2504, 338, 1444, 6588, 46314, 1358, 11, 290, 340, 318, 257, 1517, 356, 460, 466, 13, 1318, 561, 761, 284, 307, 281, 12964, 1581, 44, 20958, 2033, 286, 340, 284, 3753, 703, 881, 356, 821, 5137, 656, 262, 8137, 13, 632, 338, 4577, 284, 4646, 262, 6588, 5072, 13, 632, 561, 307, 588, 2111, 284, 12051, 5789, 748, 282, 1883, 6134, 284, 787, 4713, 1660, 780, 286, 674, 3236, 3512, 329, 10150, 75, 485, 14860, 13, 775, 460, 11, 475, 356, 6584, 470, 13, 383, 3037, 318, 44192, 1091, 276, 11, 475, 772, 611, 340, 547, 20823, 4166, 340, 561, 307, 2089, 284, 3494, 340, 13, 775, 10385, 11863, 290, 17173, 7718, 23461, 284, 7375, 17, 290, 1660, 284, 787, 2568, 13, 37941, 9482, 4898, 11, 33512, 11]
